# 07 — Built-in Functions

**What you will learn:**
- **String functions**: upper, lower, trim, length, concat, substring, split, regexp_replace, like, rlike
- **Date & Timestamp functions**: current_date, date_add, datediff, date_format, year/month/day, to_date, to_timestamp
- **Numeric / Math functions**: round, ceil, floor, abs, sqrt, rand, mod
- **Null handling functions**: coalesce, nvl, isNull / isNotNull, nullif
- **Array functions**: array, array_contains, explode, size, flatten, array_distinct, sort_array
- **Map functions**: create_map, map_keys, map_values, explode map
- **Conditional functions**: when/otherwise, case expressions

## Setup

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit

data = [
    (1, "  Alice Smith  ", "Engineering", 95000.50, "2022-01-15", None),
    (2, "bob jones",       "Marketing",   72000.00, "2021-06-01", "NYC"),
    (3, "CHARLIE BROWN",   "Engineering", 105000.0, "2020-03-20", "SF"),
    (4, "Diana Prince",    "HR",          68000.75, "2023-09-10", None),
    (5, "eve adams",       "Marketing",   78000.0,  "2022-11-05", "Chicago"),
]

df = spark.createDataFrame(data, ["id", "name", "department", "salary", "hire_date", "city"])
df.show(truncate=False)

## 1. String Functions

In [ ]:
df.select(
    col("name"),
    F.upper(col("name")).alias("upper"),
    F.lower(col("name")).alias("lower"),
    F.initcap(col("name")).alias("initcap"),          # Title Case
    F.trim(col("name")).alias("trim"),                 # remove leading/trailing spaces
    F.ltrim(col("name")).alias("ltrim"),
    F.rtrim(col("name")).alias("rtrim"),
    F.length(F.trim(col("name"))).alias("length")
).show(truncate=False)

In [ ]:
# concat and concat_ws (with separator)
df.select(
    F.concat(F.trim(col("name")), lit(" | "), col("department")).alias("name_dept"),
    F.concat_ws(" - ", col("department"), col("city")).alias("dept_city")
).show(truncate=False)

In [ ]:
# substring — extract part of a string
df.select(
    col("hire_date"),
    F.substring(col("hire_date"), 1, 4).alias("year_part"),    # pos 1, len 4
    F.substring(col("hire_date"), 6, 2).alias("month_part"),   # pos 6, len 2
    F.substring_index(col("hire_date"), "-", 1).alias("year_idx"),  # up to first "-"
).show()

In [ ]:
# split — split string into array
df.select(
    col("name"),
    F.split(F.trim(col("name")), " ").alias("name_parts"),          # split on space
    F.split(F.trim(col("name")), " ").getItem(0).alias("first_name"),
    F.split(F.trim(col("name")), " ").getItem(1).alias("last_name"),
).show(truncate=False)

In [ ]:
# regexp_replace and regexp_extract
df.select(
    col("name"),
    F.regexp_replace(col("name"), r"\s+", "_").alias("name_underscored"),  # spaces -> _
    F.regexp_extract(col("name"), r"([A-Za-z]+)\s+([A-Za-z]+)", 1).alias("first_name_regex"),
).show(truncate=False)

In [ ]:
# like (SQL LIKE pattern) and rlike (regex)
df.filter(F.trim(col("name")).rlike(r"^[A-Z]")).show(truncate=False)  # starts with uppercase

In [ ]:
# lpad / rpad — pad string to fixed length
df.select(
    col("id"),
    F.lpad(col("id").cast("string"), 5, "0").alias("padded_id")   # 1 -> 00001
).show()

In [ ]:
# locate — find position of substring (1-based, 0 = not found)
df.select(
    col("name"),
    F.locate("a", F.lower(col("name"))).alias("pos_of_a")
).show(truncate=False)

## 2. Date & Timestamp Functions

In [ ]:
from pyspark.sql.functions import (
    current_date, current_timestamp,
    to_date, to_timestamp, date_format,
    year, month, dayofmonth, dayofweek, dayofyear, quarter, weekofyear,
    date_add, date_sub, add_months, months_between, datediff, last_day,
    trunc, date_trunc
)

df_dates = df.withColumn("hire_dt", to_date(col("hire_date"), "yyyy-MM-dd"))
df_dates.printSchema()

In [ ]:
# Extract date parts
df_dates.select(
    col("hire_dt"),
    year(col("hire_dt")).alias("year"),
    month(col("hire_dt")).alias("month"),
    dayofmonth(col("hire_dt")).alias("day"),
    dayofweek(col("hire_dt")).alias("day_of_week"),  # 1=Sunday, 7=Saturday
    quarter(col("hire_dt")).alias("quarter"),
    weekofyear(col("hire_dt")).alias("week_of_year")
).show()

In [ ]:
# Date arithmetic
df_dates.select(
    col("hire_dt"),
    current_date().alias("today"),
    datediff(current_date(), col("hire_dt")).alias("days_employed"),
    date_add(col("hire_dt"), 90).alias("probation_end"),             # +90 days
    add_months(col("hire_dt"), 6).alias("six_months_later"),
    months_between(current_date(), col("hire_dt")).alias("months_employed"),
    last_day(col("hire_dt")).alias("month_end")
).show(truncate=False)

In [ ]:
# Format dates as strings
df_dates.select(
    col("hire_dt"),
    date_format(col("hire_dt"), "dd-MMM-yyyy").alias("formatted"),    # 15-Jan-2022
    date_format(col("hire_dt"), "MMMM dd, yyyy").alias("long_format") # January 15, 2022
).show()

In [ ]:
# Truncate to month / year
df_dates.select(
    col("hire_dt"),
    trunc(col("hire_dt"), "MM").alias("trunc_month"),   # first day of month
    trunc(col("hire_dt"), "YYYY").alias("trunc_year"),  # first day of year
).show()

In [ ]:
# to_timestamp — parse datetime strings
ts_data = [("2024-06-15 08:30:45",), ("2024-07-01 14:00:00",)]
df_ts = spark.createDataFrame(ts_data, ["event_time_str"])
df_ts.select(
    to_timestamp(col("event_time_str"), "yyyy-MM-dd HH:mm:ss").alias("event_time")
).show()

## 3. Numeric / Math Functions

In [ ]:
from pyspark.sql.functions import round, ceil, floor, abs, sqrt, pow, log, rand, randn, mod

df.select(
    col("salary"),
    F.round(col("salary"), 0).alias("rounded"),       # round to 0 decimals
    F.ceil(col("salary") / 1000).alias("ceil_k"),     # ceiling
    F.floor(col("salary") / 1000).alias("floor_k"),   # floor
    F.abs(col("salary") - 80000).alias("abs_diff"),   # absolute value
    F.sqrt(col("salary")).alias("sqrt_salary"),
    F.pow(col("salary"), lit(0.5)).alias("power_0.5"),
    F.mod(col("salary").cast("long"), lit(10000)).alias("salary_mod_10k")
).show()

## 4. Null Handling Functions

In [ ]:
from pyspark.sql.functions import coalesce, isnull, isnan, nanvl

df.select(
    col("name"),
    col("city"),
    coalesce(col("city"), lit("UNKNOWN")).alias("city_no_null"),  # first non-null
    isnull(col("city")).alias("city_is_null"),
).show(truncate=False)

In [ ]:
# nullif — return null if two values are equal
df.select(
    col("city"),
    F.nullif(col("city"), lit("NYC")).alias("city_no_nyc")  # null if city = NYC
).show()

## 5. Array Functions

In [ ]:
arr_data = [
    (1, ["Python", "Spark", "SQL", "Python"]),
    (2, ["SQL", "Excel"]),
    (3, ["Python", "Java", "Spark"]),
]
df_arr = spark.createDataFrame(arr_data, ["id", "skills"])
df_arr.show(truncate=False)

In [ ]:
from pyspark.sql.functions import (
    array, array_contains, array_distinct, array_sort,
    explode, explode_outer, posexplode,
    size, flatten, array_union, array_intersect, array_except,
    array_remove, sort_array
)

df_arr.select(
    col("id"),
    col("skills"),
    size(col("skills")).alias("num_skills"),
    array_contains(col("skills"), "Python").alias("knows_python"),
    array_distinct(col("skills")).alias("unique_skills"),
    sort_array(col("skills")).alias("sorted_skills"),
    array_remove(col("skills"), "Python").alias("without_python")
).show(truncate=False)

In [ ]:
# explode — one row per array element (removes rows with empty/null arrays)
df_arr.select(col("id"), F.explode(col("skills")).alias("skill")).show()

In [ ]:
# posexplode — includes position index
df_arr.select(col("id"), F.posexplode(col("skills")).alias("pos", "skill")).show()

In [ ]:
# explode_outer — keeps rows with empty/null arrays (produces null value row)
empty_arr_data = [(4, []), (5, None)]
df_empty = spark.createDataFrame(empty_arr_data, ["id", "skills"])
df_all = df_arr.unionByName(df_empty, allowMissingColumns=True)
df_all.select(col("id"), F.explode_outer(col("skills")).alias("skill")).show()

In [ ]:
# flatten — flatten nested arrays
nested = [(1, [["a", "b"], ["c"]]),
          (2, [["x"], ["y", "z"]])]
df_nested = spark.createDataFrame(nested, ["id", "nested"])
df_nested.select(col("id"), F.flatten(col("nested")).alias("flat")).show(truncate=False)

## 6. Map Functions

In [ ]:
from pyspark.sql.functions import create_map, map_keys, map_values, map_contains_key

# Create a map column from two arrays / alternating key-value cols
df_map = df.select(
    col("id"),
    F.create_map(
        lit("department"), col("department"),
        lit("city"),       F.coalesce(col("city"), lit("N/A"))
    ).alias("attributes")
)
df_map.show(truncate=False)

In [ ]:
df_map.select(
    col("id"),
    col("attributes"),
    map_keys(col("attributes")).alias("keys"),
    map_values(col("attributes")).alias("values"),
    col("attributes").getItem("department").alias("dept_from_map")
).show(truncate=False)

In [ ]:
# Explode a map — one row per key-value pair
df_map.select(col("id"), F.explode(col("attributes")).alias("key", "value")).show()

## 7. Conditional Functions: when / otherwise

In [ ]:
df.select(
    col("name"),
    col("salary"),
    F.when(col("salary") >= 100000, "Senior")
     .when(col("salary") >= 80000,  "Mid")
     .when(col("salary") >= 70000,  "Junior")
     .otherwise("Entry")
     .alias("level"),
    F.when(col("city").isNull(), "Remote").otherwise(col("city")).alias("location")
).show(truncate=False)

## Key Takeaways

| Category | Key Functions |
|---|---|
| String | `upper`, `lower`, `trim`, `concat_ws`, `split`, `regexp_replace`, `substring`, `length` |
| Date | `to_date`, `date_format`, `datediff`, `date_add`, `year/month/day`, `trunc` |
| Math | `round`, `ceil`, `floor`, `abs`, `sqrt`, `mod` |
| Null | `coalesce`, `isnull`, `nullif` |
| Array | `explode`, `array_contains`, `array_distinct`, `size`, `flatten`, `sort_array` |
| Map | `create_map`, `map_keys`, `map_values`, `explode` |
| Conditional | `when(...).when(...).otherwise(...)` |

**Import tip:** Use `import pyspark.sql.functions as F` and call `F.upper()`, `F.col()` etc.
to avoid naming conflicts with Python builtins like `sum`, `min`, `max`.